# WORD EMMBEDINGS

Referencias:
* [Vector embedding (OPENAI DEVELOPERS)](https://developers.openai.com/api/docs/guides/embeddings)
* [Similitud de cosenos (MEDIUM)](https://medium.com/geekculture/cosine-similarity-and-cosine-distance-48eed889a5c4)

In [62]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from nltk.corpus import stopwords
import nltk
from openai import OpenAI
client = OpenAI()


In [78]:
## Leer los datos
df_tortugas = (pd.read_csv('../demo_datasets/demo_tortugas.csv', sep=';')
                #.head()
                .dropna(subset=['observa']))
df_tortugas.head(n=3)

,id,ficha,especie,nombre.especie,fecha_orig,fecha,anio,mes,estacion,lugar_orig,...,fmt_lugar,muni,codmun,causa_orig,causa,muerte,observa,lesion,cuerpo,estado
1,2,1868 - 1977-1990,Caretta caretta,Tortuga Boba,14/11/1989,14/11/1989,1989,Noviembre,Otoño,NaN,...,NaN,Icod de los Vinos,38022,Cautividad,Otros,No,3 años en cautividad en agua dulce.,NaN,NaN,NaN
2,3,9537 - 1998-2010,Caretta caretta,Tortuga Boba,02/12/2010,02/12/2010,2010,Diciembre,Invierno,CANDELARIA - CANDELARIA,...,"Candelaria, Santa Cruz de Tenerife, Islas Cana...",Candelaria,38011,Enfermedad,Enfermedad,No,"Le falta la aleta delantera dcha, caparazón y ...",Herida,Varias partes,NaN
3,4,9521 - 1998-2010,Caretta caretta,Tortuga Boba,15/11/2010,15/11/2010,2010,Noviembre,Otoño,Puerto Colón,...,"Puerto, Tazacorte, Santa Cruz de Tenerife, Isl...",Adeje,38001,Artes de pesca,Artes de pesca,Si,"Corte en el cuello por enmallamiento, flaca, d...",Varias lesiones,Cuello,NaN


## Descargar stopwords y seleccionar las de español

In [52]:
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish'))
custom_words = {
    "derecha", "derecho", "dcha",
    "izquierda", "izquierdo", "izq", "izda",
    "arriba", "abajo",
    "delantera", "trasera",
    "superior", "inferior",
    "lado", "parte"
  }
all_stop_words = stop_words.union(custom_words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jcge9\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [63]:
def remove_accents_only(text):
    replacements = (
        ("á", "a"), ("é", "e"), ("í", "i"),
        ("ó", "o"), ("ú", "u"),
        ("Á", "A"), ("É", "E"), ("Í", "I"),
        ("Ó", "O"), ("Ú", "U"),
    )
    for a, b in replacements:
        text = text.replace(a, b)
    return text

def extract_words(text):
    text = remove_accents_only(text.lower())
    words = re.findall(r'\b[a-záéíóúñ]+\b', text)
    return [w for w in words if w not in all_stop_words]

## Construcción del vocabulario


In [70]:
words = []

for t in df_tortugas['observa']:
    words.extend(extract_words(t))

word_freq = Counter(words)

vocab = [w for w, f in word_freq.items() if f >= 2]

## Función para usar el modelo y obtener las embedding words

In [76]:
def get_embedding(text):
    return client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    ).data[0].embedding

word_emb = {w: get_embedding(w) for w in vocab}

## Buscar sinónimos de una palabra 

Uso de [similutud de coseno](https://medium.com/geekculture/cosine-similarity-and-cosine-distance-48eed889a5c4) con los vectores del word embedding

In [ ]:

from sklearn.metrics.pairwise import cosine_similarity

# Función que busca palabras similares (sinónimos aproximados) usando embeddings
def find_synonyms(word, word_emb, top_k=10):
    # Si la palabra no está en el diccionario de embeddings, devolvemos un mensaje
    if word not in word_emb:
        return f"'{word}' no está en el vocabulario"

    # Convierte el embedding de la palabra objetivo en un vector 2D
    target = np.array(word_emb[word]).reshape(1, -1)
    
    sims = [] # Lista donde se guardarán las similitudes

    # Recorre todas las palabras y sus embeddings
    for w, emb in word_emb.items():
        if w == word:
            continue

        # Calcula la similitud coseno entre la palabra objetivo y la actual
        sim = cosine_similarity(
            target,
            np.array(emb).reshape(1, -1)
        )[0][0]
        sims.append((w, sim)) # guarda la parabra y su similitud
    
    # Ordena las palabras por similitud de mayor a menor
    sims.sort(key=lambda x: x[1], reverse=True)
    # Devueklve las top_k palabras más similares (10 por defecto)
    return sims[:top_k]

## Ejemplos para buscar palabras relacionadas usando las word embeddings:

Nota: es cierto que, al trabajar con una demo y solo usar los primeros 100 registros, algunas palabras pueden no estar bien representadas en el diccionario, ya que no cuentan con suficiente contexto ni frecuencia. Esto hace que el vocabulario resultante sea más limitado en comparación con el que obtendríamos utilizando todo el conjunto de datos.

In [77]:
## Ejemplo de uso con palabra de lesiones corporales
palabras = ["lesión", 
            "hemorragia", 
            "fractura", 
            "contusión", 
            "corte",
            "flaca"]

for palabra in palabras:
  print(f"""
> La palabras que se buscar es: {palabra}""")
  print(find_synonyms(palabra, word_emb))


> La palabras que se buscar es: lesión
'lesión' no está en el vocabulario

> La palabras que se buscar es: hemorragia
'hemorragia' no está en el vocabulario

> La palabras que se buscar es: fractura
'fractura' no está en el vocabulario

> La palabras que se buscar es: contusión
'contusión' no está en el vocabulario

> La palabras que se buscar es: corte
[('cortes', np.float64(0.7332534932712393)), ('cuello', np.float64(0.4776447647752608)), ('amputa', np.float64(0.45515923770416883)), ('cabeza', np.float64(0.4396986413425018)), ('golpe', np.float64(0.4244860357054412)), ('amputada', np.float64(0.4052676513136529)), ('roto', np.float64(0.40117756356588763)), ('caparazon', np.float64(0.39625503731324996)), ('boca', np.float64(0.38888271690993553)), ('falta', np.float64(0.37951477187953925))]

> La palabras que se buscar es: flaca
[('boca', np.float64(0.40214909729098547)), ('falta', np.float64(0.3696971118360123)), ('llena', np.float64(0.3526651688253335)), ('inflamada', np.float64(0.34

In [51]:
## Ejemplo de uso para la palabras de causas
palabras = ["causa", 
            "pesca", 
            "rafia", 
            "plastico", 
            "enrredada",
            "golpe"]

for palabra in palabras:
  print(f"""
> La palabras que se buscar es: {palabra}""")
  print(find_synonyms(palabra, word_emb))


> La palabras que se buscar es: causa
'causa' no está en el vocabulario

> La palabras que se buscar es: pesca
'pesca' no está en el vocabulario

> La palabras que se buscar es: rafia
[('restos', np.float64(0.35407316546437717)), ('roto', np.float64(0.3536111084312737)), ('inflamación', np.float64(0.34833800472492504)), ('falta', np.float64(0.3447352718406874)), ('boca', np.float64(0.34401467742472525)), ('cortes', np.float64(0.34358615247786356)), ('problema', np.float64(0.3405177647693435)), ('flaca', np.float64(0.33857954791007583)), ('corte', np.float64(0.3353750775056702)), ('deriva', np.float64(0.3202961999424604))]

> La palabras que se buscar es: plastico
'plastico' no está en el vocabulario

> La palabras que se buscar es: enrredada
'enrredada' no está en el vocabulario

> La palabras que se buscar es: golpe
[('corte', np.float64(0.4244860357054412)), ('cuello', np.float64(0.37825260142055395)), ('cortes', np.float64(0.3770422861664644)), ('anzuelo', np.float64(0.369339080007

In [54]:
## Ejemplo de uso para la palabras partes del cuerpo
palabras = ["cuerpo", 
            "cuello", 
            "cabeza", 
            "aleta", 
            "caparazon"]

for palabra in palabras:
  print(f"""
> La palabras que se buscar es: {palabra}""")
  print(find_synonyms(palabra, word_emb))


> La palabras que se buscar es: cuerpo
'cuerpo' no está en el vocabulario

> La palabras que se buscar es: cuello
[('cabeza', np.float64(0.5867770942613237)), ('corte', np.float64(0.4776447647752608)), ('cortes', np.float64(0.47075012133962696)), ('anzuelo', np.float64(0.4021611741580977)), ('boca', np.float64(0.3903368938975516)), ('golpe', np.float64(0.37825260142055395)), ('pequeño', np.float64(0.36165082125060644)), ('caparazón', np.float64(0.3581719176660475)), ('enmallamiento', np.float64(0.3567437171626632)), ('ojo', np.float64(0.3386557377343733))]

> La palabras que se buscar es: cabeza
[('cuello', np.float64(0.5867770942613237)), ('caparazón', np.float64(0.49071615809988506)), ('boca', np.float64(0.44123825304068254)), ('corte', np.float64(0.4396986413425018)), ('cortes', np.float64(0.4161777515583074)), ('ojo', np.float64(0.39127152590106123)), ('aleta', np.float64(0.35602730235315116)), ('pequeño', np.float64(0.3555999932479129)), ('golpe', np.float64(0.35454715916908036))